# Grounded Carbon-Market QA: data + train + evaluate (Google Colab)
**Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all** (about 3 h). Keep this tab open.

All LLM work (QA generation, hallucination gate, unanswerable verification, evaluation judge) runs on a
**local Qwen2.5-14B served by Ollama on the GPU**, so there are no API limits. The question–answer pairs already
generated with Groq gpt-oss-120b on the laptop are reused from `data/work/llm_cache.jsonl`.

In [1]:
import os
!git clone -q https://github.com/snehapal0905/grounded-carbon-llm /content/proj
os.chdir('/content/proj')
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception: print('HF_TOKEN secret not set (only needed to publish to Hugging Face)')
LOCAL = 'ollama_chat/qwen2.5:14b'
os.environ.update({
    'LITELLM_LOCAL_MODEL_COST_MAP': 'True',
    'GEN_MODEL': LOCAL, 'GATE_MODEL': LOCAL, 'VERIFY_MODEL': LOCAL, 'JUDGE_MODEL': LOCAL,
    'AUDIT_JUDGE': 'litellm:' + LOCAL,
    'CACHE_FALLBACK_MODELS': 'groq/openai/gpt-oss-120b',   # reuse laptop answers
    'LLM_CONCURRENCY': '4', 'LLM_TIMEOUT': '600', 'OLLAMA_NUM_PARALLEL': '4',
})
print(sorted(os.listdir('.')), os.path.exists('data/work/chunks.jsonl'))

fatal: destination path '/content/proj' already exists and is not an empty directory.
HF_TOKEN secret not set (only needed to publish to Hugging Face)
['.env.example', '.git', '.gitignore', 'README.md', '__pycache__', 'app.py', 'common', 'config.py', 'data', 'docs', 'huggingface_tokenizers_cache', 'kaggle', 'ollama.log', 'outputs', 'requirements.txt', 'scripts', 'unsloth_compiled_cache'] True


In [2]:
!pip install -q aligntune unsloth rank_bm25 sentence-transformers litellm python-dotenv auditkit "curatorkit[all]" 2>&1 | tail -3
!python -c "import aligntune, unsloth, torch; print('ok', torch.cuda.get_device_name(0))"

W1115 19:56:23.904000 39326 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1115 19:56:23.945000 39326 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
I0000 00:00:1794772587.418838   39326 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
[aligntune] 2026-11-15 19:56:34 WARNING aligntune.core.rl.registries:54 - Could not initialize reward 'rubric' from rewards registry: Missi

In [3]:
# Local LLM server (Ollama) on the GPU
!apt-get -qq install -y zstd pciutils > /dev/null; curl -fsSL https://ollama.com/install.sh | sh > /dev/null
import subprocess, time, requests
def start_ollama():
    p = subprocess.Popen(['ollama', 'serve'], stdout=open('ollama.log', 'w'), stderr=subprocess.STDOUT)
    for _ in range(60):
        try: requests.get('http://localhost:11434'); return p
        except Exception: time.sleep(1)
ollama = start_ollama()
!ollama pull qwen2.5:14b 2>&1 | tail -1
!python -c "from common.llm import chat; import os; print(chat(os.environ['GEN_MODEL'], [{'role':'user','content':'Say OK'}], max_tokens=10))"

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
success 
OK


In [4]:
# Data steps 2-4 already finished in this runtime (outputs/ on disk); just pull the latest code
!git pull -q; ls outputs/sft; wc -l outputs/sft/train.jsonl outputs/test_candidates.csv

train.jsonl  validation.jsonl
    741 outputs/sft/train.jsonl
    151 outputs/test_candidates.csv
    892 total


In [5]:
# Free the GPU for training
ollama.terminate(); ollama.wait(); time.sleep(5)
!pkill -9 -x ollama; sleep 3; nvidia-smi --query-gpu=memory.used --format=csv

memory.used [MiB]
0 MiB


In [6]:
# Smoke test already passed in this runtime (AlignTune + Unsloth, 40 samples, 3 steps). Skipped to save GPU time.
# !python scripts/05_train.py --max-samples 40 --epochs 1

In [7]:
# Full training with AlignTune (1 epoch ~50 min on T4, fits the free Colab session)
!python scripts/05_train.py --epochs 1

W1115 19:57:56.998000 39954 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1115 19:57:57.052000 39954 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
I0000 00:00:1794772680.418151   39954 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
[aligntune] 2026-11-15 19:58:06 WARNING aligntune.core.rl.registries:54 - Could not initialize reward 'rubric' from rewards registry: Missi

In [8]:
# 3 systems on the test set
!python scripts/06_generate.py

Test set: 90 answerable, 60 unanswerable
W1115 20:42:04.392000 51597 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1115 20:42:04.446000 51597 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
Loading weights: 100% 199/199 [00:00<00:00, 831.23it/s]
Retrieval recall@4 (gold or neighbour chunk): 77.8%
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 398/398 [00:32<00:00, 12.08it/s]

== base_rag
A Suspension Decision by the Governing Board must contain the following information:

1. **The grounds for the suspension**  
2. **Sufficient information for th

In [10]:
# Backup -> fix empty fine-tuned answers -> score (grounding judge + AuditKIT)
!git pull -q
!pkill -9 -x ollama; sleep 3
!cd /content && zip -qr /content/backup.zip proj/outputs/preds proj/outputs/adapter proj/outputs/sft proj/outputs/test_candidates.csv proj/data/work/funnel.json proj/data/work/unanswerable_stats.json proj/data/work/qa_raw.jsonl proj/data/work/unanswerable.jsonl && ls -la /content/backup.zip
from google.colab import files; files.download('/content/backup.zip')
!python scripts/06b_fix_empty.py
!mkdir -p outputs/preds_unscored && mv outputs/preds/base_rag.jsonl outputs/preds_unscored/
ollama = start_ollama()
!python scripts/07_evaluate.py --no-auditkit-judge
!echo "=====RESULTS====="; cat outputs/results/report.md; echo "=====METRICS====="; cat outputs/results/metrics.json; echo "=====AKCMP====="; cat outputs/results/auditkit_compare.txt

-rw-r--r-- 1 root root 373103168 Nov 15 21:27 /content/backup.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

150 answers, 21 empty -> regenerating
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
W1115 21:27:42.652000 63708 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1115 21:27:42.707000 63708 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
Loading weights: 100% 398/398 [00:33<00:00, 11.79it/s]
still empty: 1
- [S1] (ICVCM_Core_Carbon_Principles_v3, p.129)
the Suspension Decision shall as a minimum contain: the grounds for the suspension, sufficient information for the carbon-crediting program to understand 
- The requirements for data and parameters used for quantifying re

In [11]:
# Save final scored outputs (small)
!cd /content/proj && zip -qr /content/final_results.zip outputs/results outputs/preds outputs/preds_unscored data/work/funnel.json data/work/unanswerable_stats.json && ls -la /content/final_results.zip
from google.colab import files; files.download('/content/final_results.zip')

-rw-r--r-- 1 root root 2286514 Nov 15 22:01 /content/final_results.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Demo with a public link (stop the cell to end it)
ollama.terminate()
!pip install -q gradio
!python app.py